# QueryFormer 数据处理流程演示

本notebook演示了QueryFormer项目中JSON执行计划到树结构转换以及特征编码的完整过程。

## 主要内容
1. **JSON解析**: 展示原始PostgreSQL执行计划JSON
2. **树结构转换**: 将JSON转换为树形结构
3. **特征编码**: 展示每个节点的特征向量构成
4. **图结构转换**: 转换为模型输入格式


## 1. 环境设置


In [9]:
import pandas as pd
import torch
import json
import numpy as np
import sys
import os

# 导入项目模块
from model.dataset import PlanTreeDataset
from model.database_util import get_hist_file, get_job_table_sample, Encoding
from model.util import Normalizer

print("环境设置完成")


环境设置完成


## 2. 加载数据和配置


In [10]:
# 数据路径
data_path = './data/imdb/'

# 加载训练数据（只取前5条用于演示）
print("加载训练数据...")
train_df = pd.read_csv(data_path + 'plan_and_cost/train_plan_part0.csv').head(5)
print(f"加载了 {len(train_df)} 条训练数据")

# 加载直方图文件
print("加载直方图文件...")
hist_file = get_hist_file(data_path + 'histogram_string.csv')
print(f"加载了 {len(hist_file)} 个直方图")

# 加载表采样数据
print("加载表采样数据...")
table_sample = get_job_table_sample(data_path + 'train')
print(f"加载了 {len(table_sample)} 个查询的表采样数据")


加载训练数据...
加载了 5 条训练数据
加载直方图文件...


/home/AiChaosN/Project/Phd/project/QueryFormer_VLDB2022/model/database_util.py:76: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  hist_file['freq'][i] = freq_np
/home/AiChaosN/Project/Phd/project/QueryFormer_VLDB2022/model/database_util.py:89

加载了 9 个直方图
加载表采样数据...
Loaded queries with len  100000
Loaded bitmaps
加载了 100000 个查询的表采样数据


In [11]:
# 创建编码器和标准化器
try:
    encoding_ckpt = torch.load('./checkpoints/encoding.pt', map_location='cpu')
    encoding = encoding_ckpt['encoding']
    print("从checkpoint加载编码器")
except:
    print("创建新的编码器")
    column_min_max_vals = {}
    col2idx = {'NA': 0}
    encoding = Encoding(column_min_max_vals, col2idx)

# 创建标准化器
cost_norm = Normalizer(-3.61192, 12.290855)
card_norm = Normalizer(1, 100)

print("配置创建完成")


从checkpoint加载编码器
配置创建完成


## 3. 创建数据集


In [12]:
# 创建数据集
print("创建数据集...")
dataset = PlanTreeDataset(
    json_df=train_df,
    train=None,
    encoding=encoding,
    hist_file=hist_file,
    card_norm=card_norm,
    cost_norm=cost_norm,
    to_predict='cost',
    table_sample=table_sample
)
print("数据集创建完成")


创建数据集...
数据集创建完成


## 4. 演示1: 使用数据集中的真实数据


In [13]:
# 演示第0个查询的处理过程
result1 = dataset.demo_with_sample_data(0)


使用数据集中的第 0 个样本进行演示
查询ID: 0
完整数据处理流水线演示
JSON到树结构转换演示

1. 原始JSON执行计划:
----------------------------------------
{
  "Plan": {
    "Node Type": "Gather",
    "Parallel Aware": false,
    "Startup Cost": 23540.58,
    "Total Cost": 154548.95,
    "Plan Rows": 567655,
    "Plan Width": 119,
    "Actual Startup Time": 386.847,
    "Actual Total Time": 646.972,
    "Actual Rows": 283812,
    "Actual Loops": 1,
    "Workers Planned": 2,
    "Workers Launched": 2,
    "Single Copy": false,
    "Plans": [
      {
        "Node Type": "Hash Join",
        "Parent Relationship": "Outer",
        "Parallel Aware": true,
        "Join Type": "Inner",
        "Startup Cost": 22540.58,
        "Total Cost": 96783.45,
        "Plan Rows": 236523,
        "Plan Width": 119,
        "Actual Startup Time": 369.985,
        "Actual Total Time": 518.487,
        "Actual Rows": 94604,
        "Actual Loops": 3,
        "Inner Unique": false,
        "Hash Cond": "(t.id = mi_idx.movie_id)",
        "Workers": 